In [2]:
import micropip
await micropip.install("openpyxl")

In [12]:
import pandas as pd

# ── EDIT THESE ────────────────────────────────────────────────────────────────
HISTORIC_FILE = "historic data.xlsx"          # file with the change factor sheet
FUTURE_FILE   = "future data.xlsx"            # file with the 4 future SSP sheets
OUTPUT_FILE   = "corrected future temp.xlsx"  # where results are saved

# Sheet name in your historic file that has the monthly change factors
CF_SHEET = "factors"

# The 4 sheet names in your future file (edit if yours are named differently)
FUTURE_SHEETS = [
    "sorted mid 2-4.5",
    "sorted long 2-4.5",
    "sorted mid 5-8.5",
    "sorted long 5-8.5",
]
# ─────────────────────────────────────────────────────────────────────────────

# --- 1. Load change factors --------------------------------------------------
# Expects: first column = Month (1–12), remaining columns = one per model
cf = pd.read_excel(HISTORIC_FILE, sheet_name=CF_SHEET)

# Make sure the month column is called 'Month' and is the index
cf = cf.rename(columns={cf.columns[0]: "Month"})
cf = cf.set_index("Month")
# cf now looks like:
#           ACCESS-CM2   BCC-CSM2-MR  ...
# Month
# 1          -1.97        -1.42       ...
# 2          -2.09        -1.42       ...

print("Change factor shape:", cf.shape)
print("Models found:", list(cf.columns[:5]), "...")

# --- 2. Process each future sheet -------------------------------------------
writer = pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl")

for sheet in FUTURE_SHEETS:
    print(f"\nProcessing sheet: {sheet}")

    # Load future data — first column is date, rest are model columns
    fut = pd.read_excel(FUTURE_FILE, sheet_name=sheet, header=2)
    fut = fut.rename(columns={fut.columns[0]: "date"})
    fut["date"] = pd.to_datetime(fut["date"], errors='coerce')
    fut = fut.dropna(subset=["date"])
    fut = fut.set_index("date")

    # Extract month from each row's date
    months = fut.index.month   # array of month numbers 1–12

    # Only keep model columns that exist in BOTH datasets
    common_models = [m for m in fut.columns if m in cf.columns]
    missing = [m for m in fut.columns if m not in cf.columns]
    if missing:
        print(f"  WARNING: these models are in future data but not in change "
              f"factors, skipping them: {missing}")

    fut_common = fut[common_models].copy()

    # Add the change factor: for each row, look up that month's CF per model
    # cf.loc[months] aligns each month's CF row to the corresponding date row
    cf_aligned = cf.loc[months, common_models]
    cf_aligned.index = fut_common.index   # give it the date index so addition works

    corrected = fut_common + cf_aligned

    # Reset index so date goes back as a column in the output
    corrected = corrected.reset_index()
    corrected.to_excel(writer, sheet_name=sheet, index=False)
    print(f"  Done — {len(corrected)} rows, {len(common_models)} models corrected.")

writer.close()
print(f"\nAll done. Saved to: {OUTPUT_FILE}")

Change factor shape: (12, 34)
Models found: ['ACCESS-CM2', 'ACCESS-ESM1-5', 'BCC-CSM2-MR', 'CanESM5', 'CESM2'] ...

Processing sheet: sorted mid 2-4.5
  Done — 4017 rows, 33 models corrected.

Processing sheet: sorted long 2-4.5
  Done — 4017 rows, 33 models corrected.

Processing sheet: sorted mid 5-8.5
  Done — 4017 rows, 34 models corrected.

Processing sheet: sorted long 5-8.5
  Done — 4017 rows, 34 models corrected.

All done. Saved to: corrected future temp.xlsx
